In [ ]:
# Importing Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.colors as colors
from mpl_toolkits import mplot3d
from math import sqrt
import warnings
import time

from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error
from sklearn.model_selection import train_test_split,KFold,cross_val_score,GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder,LabelEncoder, MinMaxScaler


from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor,ExtraTreesRegressor,BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV

import tensorflow as tf
import keras
from keras import layers
from keras.models import Sequential
from keras.layers import Dense
from keras.models import load_model

import pickle
import os
import random

import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (6,6)
plt.rcParams['savefig.dpi'] = 300
plt.rcParams["savefig.format"] = 'tiff'
warnings.filterwarnings("ignore")

In [ ]:
seed = 76
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [ ]:
sns.set(style='whitegrid')
sns.set_context("paper", font_scale=1.7)

In [ ]:
# Learning Rate Scheduler
def scheduler(epoch, lr):
  if epoch < 160:
    return lr
  else:
    return lr * np.exp(-0.1)

callback = keras.callbacks.LearningRateScheduler(scheduler)

# Neural Network
def Neural_network():
    init = keras.initializers.random_normal()
    model=Sequential()
    model.add(layers.Dense(64,activation='relu',kernel_initializer=init, input_dim=scaled_DF.shape[1]))
    model.add(layers.Dropout(0.1))


    model.add(layers.Dense(64,activation='relu',kernel_initializer=init))
    model.add(layers.Dropout(0.1))

    model.add(layers.Dense(1,kernel_initializer=init,activation = 'linear'))

    optimize=tf.keras.optimizers.Adam()

    model.compile(optimizer=optimize,
                    loss='mse',
                    )
    return model

In [ ]:
# Funtion to train the model
def training_model(X_train,Y_train,model):
    history=model.fit(X_train,Y_train,epochs=200,batch_size=32,verbose=0,callbacks=[callback])
    return history

In [ ]:
# To print losses
def plots():
    f, ax = plt.subplots(1,1)
    actual_test=np.array(scaler.inverse_transform(testY).reshape(-1,1))
    predicted_test=np.array(scaler.inverse_transform(model.predict(testX).reshape(-1,1)))

    #actual=testY
    #predicted= model.predict(testX)

    actual=np.array(scaler.inverse_transform(np.array(trainY).reshape(-1,1)))
    predicted=np.array(scaler.inverse_transform(model.predict(np.array(trainX)).reshape(-1,1)))

    plt.rcParams["figure.figsize"] = (6,6)
    plt.rcParams['savefig.dpi'] = 300
    plt.rcParams["savefig.format"] = 'tiff'

    sns.set(style='whitegrid')
    sns.set_context("paper", font_scale=1.7)

    plt.scatter(actual_test,predicted_test, color='purple', label='Test', linewidths=1, edgecolors='black', s=75)
    sns.regplot(x=actual,y=predicted, color='orangered', label='Train', scatter_kws={'s':40, 'alpha':0.5, 'edgecolor':'black'})

    print("Mean absolute error (MAE) - Test:      %f" % mean_absolute_error(actual_test,predicted_test))
    print("Mean squared error (MSE) - Test:       %f" % mean_squared_error(actual_test,predicted_test))
    print("Root mean squared error (RMSE) - Test: %f" % sqrt(mean_squared_error(actual_test,predicted_test)))
    print("R square (R^2):                 %f" % r2_score(actual_test,predicted_test))


    plt.xlabel("Actual")
    plt.ylabel('Predicted')
    anchored_text = AnchoredText("R\u00b2 Score_train  "+str(round(r2_score(actual,predicted),3))+'\n'"R\u00b2 Score_test  "+str(round(r2_score(actual_test,predicted_test),3)), loc=2,prop=dict(size=10))
    ax.add_artist(anchored_text)

    plt.legend(loc = 9, prop={'size': 10})
    plt.tight_layout()
    plt.savefig(str(model)[1:6], bbox_inches='tight')

In [ ]:
def defining_model(x):
    if x == 'mlr':
      model = LinearRegression()
    elif x=='adboost':
      model = AdaBoostRegressor(random_state=42)
    elif x=='xtratree':
      model = ExtraTreesRegressor(random_state=42)
    elif x=='bagging':
      model = BaggingRegressor(random_state=42)
    elif x=='pls':
      model = PLSRegression()
    elif x=='rndmfrst':
      model = RandomForestRegressor(random_state=42)
    elif x=='knn':
      model = KNeighborsRegressor()
    elif x=='svr':
      model = SVR()
    else:
      print("wrong selection")
    return model

In [ ]:
df = pd.read_excel(
    'data_final.xlsx'
)

In [ ]:
df.shape

In [ ]:
df.head(2)

In [ ]:
with open('train_ids.pkl', 'rb') as f:
    train_ids = pickle.load(f)


with open('test_ids.pkl', 'rb') as f:
    test_ids = pickle.load(f)

Train = df[df['ID'].isin(train_ids)].set_index('ID').loc[train_ids].reset_index()
Test = df[df['ID'].isin(test_ids)]

In [ ]:
# Combine Train and test for feature engineering
DF_raw = pd.concat([Train,Test],ignore_index=True)
DF_data = DF_raw.copy()

In [ ]:
DF_data.head(2)

In [ ]:
# Removing Unwanted columns
DF_data=DF_data.drop(['Type', 'ID', 'Name', 'Cation smiles', 'Anion smiles'],axis=1)
DF_data.head(3)

In [ ]:
DF_data = DF_data[['ET30', 'ATSC1c', 'An_C1SP2', 'NsOH', 'ATSC1se', 'ATSC2Z', 'ABC', 'Xpc-4d',
       'An_ETA_shape_p', 'ATSC8Z', 'An_AETA_beta', 'An_VSA_EState7',
       'An_ETA_dEpsilon_D', 'JGI8', 'AATSC2Z', 'An_SpMax_A',
       ]]

In [ ]:
DF_data.shape

In [ ]:
DF_data.head(2)

In [ ]:

# Scaling the whole DataFrame

scaler = StandardScaler()
scaled_DF = pd.DataFrame(scaler.fit_transform(DF_data.iloc[:,1:]))
scaled_DF.columns = DF_data.iloc[:,1:].columns

scaled_DF['ET30'] = scaler.fit_transform(np.array(DF_data['ET30']).reshape(-1,1))

In [ ]:
DF_target = scaled_DF[['ET30']]
scaled_DF.drop('ET30',axis=1,inplace=True)

display(scaled_DF)
display(DF_target)

In [ ]:
trainX = scaled_DF[:len(Train)]
testX = scaled_DF[len(Train):]

trainY = DF_target[:len(Train)]
testY = DF_target[len(Train):]

In [ ]:
trainX.shape

In [ ]:
testX.shape

## **Model screening with 15 descriptors**




In [ ]:
totalX = pd.concat([trainX, testX], axis=0)
totalY = pd.concat([trainY, testY], axis=0)

In [ ]:
# Extra Tree Regressor
model = defining_model(x = 'xtratree')
kfold = KFold(n_splits=5, shuffle=True)

scores = []
rmse = []
for train,valid in kfold.split(totalX,totalY):
  model.fit(totalX.iloc[train],totalY.iloc[train])
  scores.append(model.score(totalX.iloc[valid],totalY.iloc[valid]))
  actual = totalY.iloc[valid]
  predicted = model.predict(totalX.iloc[valid])
  rmse.append(sqrt(mean_squared_error(scaler.inverse_transform(actual),scaler.inverse_transform(predicted.reshape(-1,1)))))

print("Average validation R2 score after crossvalidation : ", np.mean(scores))
print("Average validation rmse score after crossvalidation : ", np.mean(rmse))


# Train model on whole train data
model = defining_model(x = 'xtratree')
model.fit(trainX,trainY)
print("\n\nTraining Accuracy : ",model.score(trainX,trainY)) # Training Accuracy
plots()

In [ ]:
# Random Forest
model = defining_model(x = 'rndmfrst')
kfold = KFold(n_splits=5, shuffle=True)

scores = []
rmse = []
for train,valid in kfold.split(totalX,totalY):
  model.fit(totalX.iloc[train],totalY.iloc[train])
  scores.append(model.score(totalX.iloc[valid],totalY.iloc[valid]))
  actual = totalY.iloc[valid]
  predicted = model.predict(totalX.iloc[valid])
  rmse.append(sqrt(mean_squared_error(scaler.inverse_transform(actual),scaler.inverse_transform(predicted.reshape(-1,1)))))

print("Average validation R2 score after crossvalidation : ", np.mean(scores))
print("Average validation rmse score after crossvalidation : ", np.mean(rmse))


# Train model on whole train data
model = defining_model(x = 'rndmfrst')
model.fit(trainX,trainY)
print("\n\nTraining Accuracy : ",model.score(trainX,trainY)) # Training Accuracy
plots()

In [ ]:
model = Neural_network()

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

scores = []
rmse = []
for train,valid in kfold.split(totalX,totalY):
  model = Neural_network()
  training_model(totalX.iloc[train],totalY.iloc[train],model)
  scores.append(r2_score(totalY.iloc[valid],model.predict(totalX.iloc[valid])))

  actual = totalY.iloc[valid]
  predicted = model.predict(totalX.iloc[valid])
  rmse.append(sqrt(mean_squared_error(scaler.inverse_transform(actual),scaler.inverse_transform(predicted))))

print("Average validation R2 score after crossvalidation : ", np.mean(scores))
print("Average validation rmse score after crossvalidation : ", np.mean(rmse))

# # Train model on whole train data
tf.keras.backend.clear_session()
model = Neural_network()
training_model(trainX,trainY,model)

actual=np.array(scaler.inverse_transform(np.array(trainY)))
predicted=np.array(scaler.inverse_transform(model.predict(np.array(trainX)).reshape(-1,1)))
model.save('nn.keras')
score = r2_score(actual,predicted)
print("\n\nTraining Accuracy : ",score) # Training Accuracy
plots()